**Setup and configuration**

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

ENV_FILE = Path.cwd() / "RAG.env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Environment file not found: {ENV_FILE}")

load_dotenv(ENV_FILE, override=True)

PDF_PATH = Path(os.environ["PDF_PATH"])
DATA_DIR = PDF_PATH.parent
PROJECT_DIR = PDF_PATH.parent.parent

QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_TIMEOUT = int(os.environ["QDRANT_TIMEOUT"])
COLLECTION_NAME = os.environ["COLLECTION_NAME"]

DENSE_MODEL_NAME = os.environ["DENSE_MODEL_NAME"]
SPARSE_MODEL_NAME = os.environ["SPARSE_MODEL_NAME"]

LLM_BASE_URL = os.environ["LLM_BASE_URL"]
LLM_MODEL = os.environ["LLM_MODEL"]
LLM_API_KEY = os.environ["LLM_API_KEY"]
LLM_TIMEOUT = int(os.environ["LLM_TIMEOUT"])
LLM_MAX_RETRIES = int(os.environ["LLM_MAX_RETRIES"])

TOP_K = int(os.environ["TOP_K"])
BATCH_SIZE = int(os.environ["BATCH_SIZE"])

print(f"Loaded settings from {ENV_FILE}")
print(f"PDF: {PDF_PATH}")

In [ ]:
%pip install -U langfuse python-dotenv

In [ ]:
import langchain_core
import langfuse

print("LangChain Core:", langchain_core.__version__)
print("Langfuse:", langfuse.__version__)

In [ ]:
from dotenv import load_dotenv
from langfuse import get_client
from langfuse.langchain import CallbackHandler

load_dotenv()

langfuse = get_client()
langfuse_handler = CallbackHandler()

print("Langfuse initialized successfully")

In [ ]:
import sys
import os
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("GPU available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed.")

Version checking for Langchain and Langgraph 

In [ ]:
import importlib.metadata
import langchain
import langgraph
import langchain_text_splitters

print("LangChain:", langchain.__version__)
try:
    print("LangGraph:", importlib.metadata.version("langgraph"))
except importlib.metadata.PackageNotFoundError:
    print("LangGraph: version unknown")
print("Core imports successful.")

Path for the input document 

In [ ]:

 # Replace with your actual PDF file name"

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

print(f"Using PDF: {PDF_PATH}")

Checks the directory_Path exists?

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Directory exists:", PROJECT_DIR.exists())

Configuring  Vector database

Imported and intialized the Dense model:MiniLM-L6-v2 and Sparse Model:Qdrant/bm25 , whereas Qdrant for vector storage 

In [ ]:

from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding

# Embedding models
dense_model = TextEmbedding(
DENSE_MODEL_NAME,
)

sparse_model = SparseTextEmbedding(SPARSE_MODEL_NAME)

# Connect to the migrated Qdrant Server
client = QdrantClient(
    url=QDRANT_URL,
    timeout=QDRANT_TIMEOUT,
)

# Verify the migrated collection
if not client.collection_exists(COLLECTION_NAME):
    raise RuntimeError(
        f"Collection {COLLECTION_NAME} not found."
    )

info = client.get_collection(COLLECTION_NAME)

count = client.count(
    collection_name=COLLECTION_NAME,
    exact=True,
).count

print("Qdrant Server connected successfully.")
print("Collection:", COLLECTION_NAME)
print("Points:", count)
print("Collection status:", info.status)


In [ ]:

info = client.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Points:", info.points_count)
print("Status:", info.status)

print("\nDense configuration:")
print(info.config.params.vectors)

print("\nSparse configuration:")
print(info.config.params.sparse_vectors)

points, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=1,
    with_payload=True,
    with_vectors=True,
)

if points:
    print("\nSample point ID:", points[0].id)
    print("Vector names:", list(points[0].vector.keys()))
    print("Payload keys:", list(points[0].payload.keys()))


Implmenting Hybrid search (Dnese + Sparse) , Dense Model get's tp 20 search , Sparse Model get's top 20 search and later used the RRF that combines the ranking from multiple retrivelmethods into a single ranking 

In [ ]:
from qdrant_client import models    # This implments the hybrid search ( dense + sparse )

# Dense model is loaded from RAG.env.

def hybrid_search(query: str, limit: int = TOP_K):
    if not query.strip():
        raise ValueError("Query cannot be empty.")

    if limit < 1:
        raise ValueError("limit must be at least 1.")

    dense_vector = list(dense_model.query_embed(query))[0]
    sparse_vector = list(sparse_model.query_embed(query))[0]

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(
                query=dense_vector.tolist(),
                using="dense",
                limit=max(limit * 4, 20), #finds the top 20 similar vector embeddings
            ),
            models.Prefetch(
                query=models.SparseVector(
                    indices=sparse_vector.indices.tolist(),
                    values=sparse_vector.values.tolist(),
                ),
                using="sparse",
                limit=max(limit * 4, 20),  # finds the tp 20 keyword /lexixcal
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF), # finds the top 5 most similar ranked
        limit=limit,
        with_payload=True,
    )

    return [
        {
            "id": point.id,
            "score": point.score,
            "text": point.payload.get("text", ""),
            "metadata": {
                key: value
                for key, value in point.payload.items()
                if key != "text"
            },
        }
        for point in results.points
    ]

print("Hybrid search function defined.")

**Contextulization Retrivel**

In [ ]:
def contextualize_chunk( # This is the contextulization Retrivel for context fo every chuncks made with adding the source as page no.
    chunk_text: str,
    document_title: str,
    section_title: str,
    page_no: int,
) -> dict:
    if not chunk_text.strip():
        raise ValueError("Chunk text cannot be empty.")

    if page_no < 1:
        raise ValueError("page_no must be a positive page number.")

    context = (
        f"Document: {document_title}\n"
        f"Section: {section_title}\n"
        f"Source: page no. {page_no}"
    )

    contextualized_text = (
        f"{context}\n\n"
        f"Chunk content:\n{chunk_text}"
    )

    return {
        "text": chunk_text,
        "context": context,
        "contextualized_text": contextualized_text,
        "source_page": page_no,
        "document_title": document_title,
        "section_title": section_title,
    }

Setting the chunk size limit    

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

print("Chunk size:", CHUNK_SIZE)
print("Chunk overlap:", CHUNK_OVERLAP)
print("Text splitter configured.")

Creates the Page chunk 

In [ ]:
def create_page_chunks(pages, document_title, section_title=""):
    """
    pages: list of dictionaries with:
        {"page_no": int, "text": str}
    """
    chunks = []

    for page in pages:
        page_no = page["page_no"]
        page_text = page["text"]

        if not page_text.strip():
            continue

        page_chunks = text_splitter.split_text(page_text)

        for chunk_index, chunk_text in enumerate(page_chunks):
            chunk = contextualize_chunk(
                chunk_text=chunk_text,
                document_title=document_title,
                section_title=section_title,
                page_no=page_no,
            )

            chunk["chunk_index"] = chunk_index
            chunks.append(chunk)

    return chunks

Generating the Dense and sparse Vectors , storing them into Qdrant 

In [ ]:
from qdrant_client import models         # Generating Dense and sparse vectors and storing them into the Qrdrant

def index_chunks(points, batch_size=BATCH_SIZE):
    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")

    for start in range(0, len(points), batch_size):
        batch = points[start:start + batch_size]

        texts = [point["contextualized_text"] for point in batch]

        dense_vectors = list(dense_model.embed(texts))
        sparse_vectors = list(sparse_model.embed(texts))

        qdrant_points = []

        for point, dense_vector, sparse_vector in zip(
            batch, dense_vectors, sparse_vectors
        ):
            qdrant_points.append(
                models.PointStruct(
                    id=point["id"],
                    vector={
                        "dense": dense_vector.tolist(),
                        "sparse": models.SparseVector(
                            indices=sparse_vector.indices.tolist(),
                            values=sparse_vector.values.tolist(),
                        ),
                    },
                    payload={
                        "text": point["text"],
                        "context": point["context"],
                        "contextualized_text": point["contextualized_text"],
                        "source_page": point["source_page"],
                        "document_title": point["document_title"],
                        "section_title": point["section_title"],
                        "chunk_index": point["chunk_index"],
                    },
                )
            )

        client.upsert(
            collection_name=COLLECTION_NAME,
            points=qdrant_points,
            wait=True,
        )

        print(f"Indexed {min(start + batch_size, len(points))}/{len(points)} chunks")

    print("Indexing complete.")

Generates the deterministic ID for each document chunk before it is stored in Qdrant

In [ ]:
import hashlib

def prepare_qdrant_points(chunks):
    points = []

    for chunk in chunks:
        stable_key = (
            f"{chunk['document_title']}|"
            f"{chunk['source_page']}|"
            f"{chunk['chunk_index']}|"
            f"{chunk['text']}"
        )

        point_id = hashlib.sha256(
            stable_key.encode("utf-8")
        ).hexdigest()[:32]

        points.append({
            "id": point_id,
            "text": chunk["text"],
            "context": chunk["context"],
            "contextualized_text": chunk["contextualized_text"],
            "source_page": chunk["source_page"],
            "document_title": chunk["document_title"],
            "section_title": chunk["section_title"],
            "chunk_index": chunk["chunk_index"],
        })

    return points




# Contextualize the paragraph

# Add the chunk index required for stable IDs


# Prepare and index the point


**Inserting the pdf and later chuncking it**

In [ ]:
from pathlib import Path
import pymupdf

# Convert string to a Path object to enable .is_file() and .name properties

if not PDF_PATH.is_file():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

def extract_pdf_pages(pdf_path):
    pages = []

    with pymupdf.open(pdf_path) as pdf:         # extracted the text and made the chucks  of it
        for page_number, page in enumerate(pdf, start=1):
            pages.append({
                "page_no": page_number,
                "text": page.get_text("text").strip(),
            })

    return pages

pages = extract_pdf_pages(PDF_PATH)

print("PDF:", PDF_PATH.name)
print("Total pages:", len(pages))
print("Pages with extracted text:", sum(bool(p["text"]) for p in pages))
print("\nFirst page preview:\n", pages[0]["text"][:1000] if pages else "No pages extracted.")

In [ ]:
document_title = PDF_PATH.stem

pdf_chunks = create_page_chunks(
    pages=pages,
    document_title=document_title,
    section_title="",
)

print("Document:", document_title)
print("Pages extracted:", len(pages))
print("Chunks created:", len(pdf_chunks))

print("\nFirst chunk:")
print("Source: page no.", pdf_chunks[320]["source_page"])
print(pdf_chunks[320]["contextualized_text"][:1000])

In [ ]:
pdf_points = prepare_qdrant_points(pdf_chunks)

print("Prepared points:", len(pdf_points))
print("Unique IDs:", len({point["id"] for point in pdf_points}))
print("\nFirst point:")
print("ID:", pdf_points[0]["id"])
print("Document:", pdf_points[0]["document_title"])
print("Source: page no.", pdf_points[0]["source_page"])
print("Text preview:", pdf_points[0]["text"][:200])

In [ ]:
# Index the prepared PDF chunks in batches
BATCH_SIZE = int(os.environ["BATCH_SIZE"])

print("Document:", PDF_PATH.name)
print("Chunks to index:", len(pdf_points))
print("Batch size:", BATCH_SIZE)
from qdrant_client import models

if not pdf_points:
    raise ValueError("No PDF chunks were prepared; keeping the existing index.")

document_titles = {point["document_title"] for point in pdf_points}
if len(document_titles) != 1:
    raise ValueError(
        f"Expected chunks for one document, found: {document_titles}"
    )

document_title = next(iter(document_titles))

print("Document:", document_title)
print("Chunks to index:", len(pdf_points))
print("Batch size:", BATCH_SIZE)

client.delete(
    collection_name=COLLECTION_NAME,
    points_selector=models.FilterSelector(
        filter=models.Filter(
            must=[
                models.FieldCondition(
                    key="document_title",
                    match=models.MatchValue(value=document_title),
                )
            ]
        )
    ),
    wait=True,
)
index_chunks(pdf_points, batch_size=BATCH_SIZE)

This is for testing the input 

In [ ]:
query = "What is the MKC?"

results = hybrid_search(query, limit=5)

for rank, result in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Score:", round(result["score"], 4))
    print("Document:", result["metadata"].get("document_title"))
    print("Source: page no.", result["metadata"].get("source_page"))
    print("Text:", result["text"][:600])

**Retrival Tool**

In [ ]:
from langchain_core.tools import tool

@tool
def search_documents(query: str) -> str:
    """Search indexed documents using dense and sparse hybrid retrieval.
    Use this tool when an answer requires evidence from the document corpus.
    """
    results = hybrid_search(query, limit=5)

    if not results:
        return "No relevant document passages were found."

    formatted_results = []

    for rank, result in enumerate(results, start=1):
        metadata = result["metadata"]

        formatted_results.append(
            f"Result {rank}\n"
            f"Document: {metadata.get('document_title', 'Unknown')}\n"
            f"Source: page no. {metadata.get('source_page', 'Unknown')}\n"
            f"Content:\n{result['text']}"
        )

    return "\n\n---\n\n".join(formatted_results)

print("Tool name:", search_documents.name)
print("Description:", search_documents.description)

**Agent Inspection tool**

In [ ]:
from langchain_core.tools import tool

@tool
def list_indexed_documents() -> str:
    """List documents currently indexed in Qdrant, with their chunk counts and source pages."""
    document_stats = {}
    offset = None

    while True:
        points, offset = client.scroll(
            collection_name=COLLECTION_NAME,
            offset=offset,
            limit=256,
            with_payload=[
                "document_title",
                "source_page",
            ],
            with_vectors=False,
        )

        for point in points:
            payload = point.payload
            title = payload.get("document_title", "Unknown")
            page_no = payload.get("source_page")

            if title not in document_stats:
                document_stats[title] = {
                    "chunks": 0,
                    "pages": set(),
                }

            document_stats[title]["chunks"] += 1

            if page_no is not None:
                document_stats[title]["pages"].add(page_no)

        if offset is None:
            break

    if not document_stats:
        return "No documents are currently indexed."

    lines = []

    for title, stats in sorted(document_stats.items()):
        lines.append(
            f"Document: {title}\n"
            f"Chunks: {stats['chunks']}\n"
            f"Distinct source pages: {len(stats['pages'])}"
        )

    return "\n\n".join(lines)


print("Tool name:", list_indexed_documents.name)
print("Description:", list_indexed_documents.description)

Registered both the tool

In [ ]:
rag_tools = [
    search_documents,
    list_indexed_documents,
]

tool_registry = {
    rag_tool.name: rag_tool
    for rag_tool in rag_tools
}

print("Registered tools:")
for name in tool_registry:
    print("-", name)

**Defining all the prompts**

In [ ]:
ROUTER_SYSTEM_PROMPT = """

You are a query classification router.

RETRIEVE: Questions requiring information from the indexed DPR documents.
DIRECT: General knowledge, greetings, and unrelated questions.

Return exactly one label: RETRIEVE or DIRECT.

You are a query router for a document-grounded RAG system.

Decide whether the user's question requires searching the indexed documents.

Choose RETRIEVE when:
- The user asks about facts, procedures, or details in the documents.
- The answer requires evidence from the indexed corpus.
- The question refers to a document, page, section, or indexed source.

Choose DIRECT when:
- The user asks a general conversational question.
- The request can be handled without consulting the indexed documents.
Return only one label: RETRIEVE or DIRECT.
"""

AGENT_SYSTEM_PROMPT = """

TOOL SELECTION RULES:

1. Use search_documents when the user asks about:
   - Project objectives
   - Project details
   - Technical specifications
   - DPR content
   - Any factual information contained in documents.

2. Use list_indexed_documents only when the user asks:
   - Which documents are indexed?
   - What documents are available?
   - List the available documents.

3. Never use list_indexed_documents to answer
   questions about document content.

4. Do not answer document-specific questions
   without retrieving supporting information.
   
You are a document-grounded research assistant.

Rules:
1. Use search_documents when document evidence is needed.
2. Use list_indexed_documents when the user asks what documents are indexed.
3. Base document-specific claims on retrieved source text.
4. Cite sources using the document title and 'page no. X'.
5. Never invent source pages, quotations, or document facts.
6. If the retrieved evidence is insufficient, state that clearly.
7. Distinguish source evidence from general knowledge.
8. Provide concise, direct answers.
"""

print("Router prompt defined:", bool(ROUTER_SYSTEM_PROMPT.strip()))
print("Agent prompt defined:", bool(AGENT_SYSTEM_PROMPT.strip()))

**Graph state**

In [ ]:
from typing import Literal
from typing_extensions import NotRequired
from langgraph.graph import MessagesState

class RAGState(MessagesState):
    query: NotRequired[str]
    route: NotRequired[Literal["RETRIEVE", "DIRECT"]]
    phase: NotRequired[Literal["ROUTE", "AGENT", "ANSWER", "DONE"]]

**Attaching the LLM**

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
    temperature=0.2,
    timeout=LLM_TIMEOUT,
    max_retries=LLM_MAX_RETRIES
)

print("LangChain LLM initialized!")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
                                                                                   #Testing the LLM
response = llm.invoke([
    SystemMessage(content="You are a concise technical assistant."),
    HumanMessage(content="Reply with exactly: LLM connection successful"),
])

print(response.content)

In [ ]:
from langchain_core.tools import tool                      # Tesing caling the tool

@tool
def test_rag_tool(query: str) -> str:
    """Test tool that accepts a query and returns a fixed confirmation."""
    return f"Tool received query: {query}"

llm_with_tools = llm.bind_tools([test_rag_tool])

tool_test_response = llm_with_tools.invoke([
    SystemMessage(
        content=(
            "You are a document research assistant. "
            "You must call test_rag_tool exactly once "
            "to process the user's document-related request."
        )
    ),
    HumanMessage(content="Search the document for information about its objectives, sources.")
])

print("Tool calls:", tool_test_response.tool_calls)

In [ ]:
llm_with_rag_tools = llm.bind_tools(rag_tools)
                                                        # BINDING ALL THE TOOLS


In [ ]:
tool_response = llm_with_rag_tools.invoke([
    SystemMessage(content=AGENT_SYSTEM_PROMPT),
    HumanMessage(
        content="Find information about the project objectives in the DPR document."
    )
])

print("Tool calls:", tool_response.tool_calls)
print("Response:", tool_response.content)

In [ ]:
from langchain_core.messages import ToolMessage

tool_call = tool_response.tool_calls[0]

selected_tool = next(
    tool for tool in rag_tools
    if tool.name == tool_call["name"]
)

tool_result = selected_tool.invoke(tool_call["args"])

print("Retrieved document content:")
print(tool_result)

Implmentation of router node 

In [ ]:

from langchain_core.messages import SystemMessage, HumanMessage


def router_node(state: RAGState):

    phase = state.get("phase", "ROUTE")

    # Stage 1: Classify user query
    if phase == "ROUTE":

        query = state["messages"][-1].content

        response = llm.invoke([
            SystemMessage(content=ROUTER_SYSTEM_PROMPT),
            HumanMessage(content=query),
        ])

        decision = response.content.strip().upper()

        if decision == "RETRIEVE":
            return {
                "query": query,
                "route": "RETRIEVE",
                "phase": "AGENT",
            }

        elif decision == "DIRECT":
            answer = llm.invoke([
                SystemMessage(
                    content="You are a helpful AI assistant.Answer the user's question accurately and clearly.If the question requires information from documents,use the retrieved context when available."
                ),
                HumanMessage(content=query),
            ])

            return {
                "query": query,
                "route": "DIRECT",
                "phase": "DONE",
                "messages": [answer],
            }

        else:
            raise ValueError(
                f"Router returned an invalid decision: {response.content}"
            )

    # Stage 2: Generate final answer from retrieved evidence
    elif phase == "ANSWER":

        response = llm.invoke([
            SystemMessage(
                content=(
                    AGENT_SYSTEM_PROMPT
                    + "\n\nUse the tool results in the conversation "
                    "to answer the original user query. "
                    "Do not call tools again. "
                    "Do not invent information absent from the retrieved evidence. "
                    "Preserve relevant document page references."
                )
            ),
            *state["messages"],
        ])

        return {
            "phase": "DONE",
            "messages": [response],
        }

    else:
        raise ValueError(
            f"Invalid router phase: {phase}"
        )


Agent node 

In [ ]:
def agent_node(state: RAGState):

    response = llm_with_rag_tools.invoke(
        [
            SystemMessage(content=AGENT_SYSTEM_PROMPT),
            *state["messages"]
        ]
    )

    return {
        "messages": [response],
        "phase": "AGENT" if response.tool_calls else "ANSWER"
    }

Conditional Routing 

In [ ]:
from langgraph.graph import END

def route_after_router(state: RAGState):

    if state["phase"] == "AGENT":
        return "agent"

    return END

In [ ]:
from langgraph.prebuilt import tools_condition, ToolNode

Declaring the Graph node 

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage

def prepare_node(state: RAGState):

    latest_query = next(
        msg.content
        for msg in reversed(state["messages"])
        if isinstance(msg, HumanMessage)
    )

    return {
        "query": latest_query,
        "route": "DIRECT",  # Temporary default; router overwrites it.
        "phase": "ROUTE"
    }


# Build graph
graph_builder = StateGraph(RAGState)

graph_builder.add_node("prepare", prepare_node)
graph_builder.add_node("router", router_node)
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", ToolNode(rag_tools))

# Entry
graph_builder.add_edge(START, "prepare")
graph_builder.add_edge("prepare", "router")

# Router -> Agent or End
graph_builder.add_conditional_edges(
    "router",
    route_after_router,
    {
        "agent": "agent",
        END: END
    }
)

# Agent -> Tools or Router
graph_builder.add_conditional_edges(
    "agent",
    tools_condition,
    {
        "tools": "tools",
        END: "router"
    }
)

# Tool execution -> Agent
graph_builder.add_edge("tools", "agent")

# Compile with conversation memory
memory = MemorySaver()

rag_agent = graph_builder.compile(
    checkpointer=memory
)

print("RAG Agent compiled successfully!")

Visualization of graph 

In [ ]:
from IPython.display import Image, display

display(
    Image(rag_agent.get_graph().draw_mermaid_png())
)

Testing the complete agent 

In [ ]:
import uuid
from langchain_core.messages import HumanMessage

# Run once to start a new conversation; reuse this ID for its follow-up questions.
conversation_id = str(uuid.uuid4())

def chat_with_agent(query: str):
    config = {
        "configurable": {
            "thread_id": conversation_id
        },
        "recursion_limit": 20
    }

    result = rag_agent.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=config
    )

    return result["messages"][-1].content

In [ ]:
from langchain_core.messages import HumanMessage

result = rag_agent.invoke(
    {"messages": [HumanMessage(content="what is GenAI?")]},
    config={
        "configurable": {"thread_id": "test-direct"},
        "callbacks": [langfuse_handler]
    }
)

print("Agent response:")
print(result["messages"][-1].content)